# 01 · TCGA Intake & QC (Beginner‑Friendly)

**Goal:** Load TCGA‑COAD files (counts, clinical, MAF, methylation, TCMA microbiome) using the manifest.
Run simple checks: shapes, missing values, sample ID overlaps, and basic plots.

👉 Tips:
- If a cell prints a **TODO**, read it and fix your input files or manifest.
- Don’t edit the code unless you know what you’re doing—just fix files/paths.


In [ ]:

# === 🔧 SETTINGS (you only edit the manifest CSVs) ===
BASE = "/content/drive/MyDrive/Colorectal_Hippo_Dysbiosis"
MANIFEST = f"{BASE}/config/manifest_tcga.csv"
print("Using manifest:", MANIFEST)


In [ ]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt, re
from pathlib import Path

def read_manifest(path):
    df = pd.read_csv(path)
    assert set(["key","path"]).issubset(df.columns), "Manifest must have columns: key,path"
    d = df.set_index("key")["path"].to_dict()
    expected = ["counts","clinical","maf","methylation_450k","tcma_microbiome"]
    missing = [k for k in expected if k not in d]
    if missing:
        print("TODO: Add missing keys to manifest:", missing)
    return d

def guess_counts_orientation(df):
    tmp = df.copy()
    if tmp.columns[0].lower() in ["gene","gene_id","symbol"]:
        tmp = tmp.set_index(tmp.columns[0])
    if tmp.shape[0] > tmp.shape[1] and np.issubdtype(tmp.dtypes.min(), np.number):
        return 'genes_by_cols', tmp.T
    return 'genes_by_rows', tmp

def clean_sample_ids(idx):
    return pd.Index(idx.astype(str).str.strip().str.replace(r'[^A-Za-z0-9_\-\.]+','_', regex=True))


In [ ]:

man = read_manifest(MANIFEST)
man


In [ ]:

print("\n=== Loading counts ===")
counts_raw = pd.read_csv(man["counts"], sep=None, engine="python")
orient, counts = guess_counts_orientation(counts_raw)
print("Orientation:", orient)
print("Counts shape (genes x samples):", counts.shape)
display(counts.head())

# QC
print("\nCounts QC:")
print("Min value:", counts.min().min())
print("Total NA:", counts.isna().sum().sum())
dup_genes = counts.index[counts.index.duplicated()].unique()
if len(dup_genes):
    print(f"TODO: {len(dup_genes)} duplicated gene IDs; consider aggregating by sum.")


In [ ]:

libsize = counts.sum(axis=0)
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.hist(libsize, bins=30)
plt.title("Library size per sample (TCGA)")
plt.xlabel("Total counts"); plt.ylabel("Samples")
plt.tight_layout(); plt.show()


In [ ]:

print("\n=== Clinical intake ===")
clinical = pd.read_csv(man["clinical"])
print("Clinical shape:", clinical.shape)
display(clinical.head())

cand = [c for c in clinical.columns if re.search("sample|barcode|submitter", c, flags=re.I)]
print("Candidate sample ID columns:", cand)
id_col = cand[0] if cand else clinical.columns[0]

# clean ids & overlap
counts.columns = clean_sample_ids(counts.columns)
clinical[id_col] = clean_sample_ids(clinical[id_col])
overlap = sorted(set(counts.columns) & set(clinical[id_col]))
print(f"Overlap samples: {len(overlap)}")


In [ ]:

print("\n=== Optional files preview ===")
for k in ["maf","methylation_450k","tcma_microbiome"]:
    if k in man:
        try:
            df = pd.read_csv(man[k], sep=None, engine="python", nrows=5)
            print(f"{k} preview:")
            display(df.head(3))
        except Exception as e:
            print(f"NOTE: Could not preview {k} ->", e)


In [ ]:

counts.to_csv(f"{BASE}/data_processed/tcga_counts_clean.csv")
pd.Series(overlap, name="sample_id").to_csv(f"{BASE}/results/qc/tcga_samples_intersection.csv", index=False)
print("Saved -> data_processed/tcga_counts_clean.csv")
print("Saved -> results/qc/tcga_samples_intersection.csv")
print("\nNEXT → Run 02 (IBD QC) then 03 (Normalization).")
